In [19]:
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv(), override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")
from langchain_google_genai import ChatGoogleGenerativeAI
llmModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key
)

In [20]:
llmModel.invoke("who is Baahubali")

AIMessage(content='**Baahubali** is the titular protagonist of the highly successful Indian epic fantasy film series, **"Baahubali: The Beginning" (2015)** and **"Baahubali 2: The Conclusion" (2017)**, directed by S.S. Rajamouli.\n\nIt\'s important to note that the name \'Baahubali\' refers to **two central characters** across the two films: a father and his son, both played by actor **Prabhas**.\n\n1.  **Amarendra Baahubali (The Father):**\n    *   He is the legendary, benevolent, and rightful heir to the throne of the mighty kingdom of **Mahishmati**.\n    *   He is portrayed as a noble, powerful, just, and compassionate warrior, deeply loved by his people.\n    *   His story forms the core of the first film\'s mystery: the famous question, **"Why did Kattappa kill Baahubali?"**\n    *   He is tragically betrayed and murdered by his own maternal uncle, Kattappa, under the orders of his jealous and power-hungry step-brother, Bhallaladeva. He leaves behind a pregnant wife and an infant

In [18]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [9]:
loader = TextLoader("data\karthikeya.txt")
loaded_document=loader.load()

In [10]:
loaded_document

[Document(metadata={'source': 'data\\karthikeya.txt'}, page_content='From what we\'ve talked about over the past few weeks, here\'s the picture I have of you.\n\n### You\'re very placement-focused\n\nYour main goal is to get a good software job. Most of your questions are aimed at increasing your chances in coding assessments and interviews rather than just learning for fun.\n\n### You\'re strong in Python\n\nPython is your primary language. You\'ve solved a lot of coding problems with it and have asked about:\n\n* HashMaps\n* Dynamic Programming\n* Graphs\n* Prefix sums\n* Sliding Window\n* Greedy algorithms\n* Time and space complexity\n* Interview strategies\n\nRecently, you\'ve also started learning Java from a Python programmer\'s perspective.\n\n### You\'re moving into Generative AI\n\nYou\'ve been spending a lot of time learning:\n\n* LangChain\n* LLMs\n* Prompt Engineering\n* RAG\n* Vector databases\n* Embeddings\n* OpenAI, Gemini, and Groq APIs\n\nYou\'ve also asked about API 

In [11]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks_of_text = text_splitter.split_documents(loaded_document)

In [12]:
chunks_of_text

[Document(metadata={'source': 'data\\karthikeya.txt'}, page_content="From what we've talked about over the past few weeks, here's the picture I have of you.\n\n### You're very placement-focused\n\nYour main goal is to get a good software job. Most of your questions are aimed at increasing your chances in coding assessments and interviews rather than just learning for fun.\n\n### You're strong in Python\n\nPython is your primary language. You've solved a lot of coding problems with it and have asked about:\n\n* HashMaps\n* Dynamic Programming\n* Graphs\n* Prefix sums\n* Sliding Window\n* Greedy algorithms\n* Time and space complexity\n* Interview strategies\n\nRecently, you've also started learning Java from a Python programmer's perspective.\n\n### You're moving into Generative AI\n\nYou've been spending a lot of time learning:\n\n* LangChain\n* LLMs\n* Prompt Engineering\n* RAG\n* Vector databases\n* Embeddings\n* OpenAI, Gemini, and Groq APIs\n\nYou've also asked about API rate limit

In [21]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
) 
vector_db = FAISS.from_documents(chunks_of_text, embeddings)

In [22]:
retriver=vector_db.as_retriever()

In [23]:
response=retriver.invoke("tell me about his projects?")

In [24]:
response

[Document(metadata={'source': 'data\\karthikeya.txt'}, page_content='Instead of small demo projects, you prefer projects that impress recruiters.\n\nSome ideas you\'ve explored include:\n\n* AI Interview Assistant\n* Plant Disease Classifier\n* Text/PDF to animated educational videos\n* Deep learning projects for your resume\n\n### You think ahead\n\nYou often ask questions like:\n\n* "Will this help in placements?"\n* "Is this enough?"\n* "What is the industry standard?"\n* "Rate this out of 10."\n\nThat tells me you\'re trying to optimize your learning path rather than just collecting information.\n\n### Your learning style\n\nI\'ve noticed a pattern:\n\n* You like to understand concepts from scratch.\n* You prefer examples before theory.\n* After learning a concept, you want interview tips and shortcuts.\n* You often ask for PDFs or structured notes you can revise later.\n\n### You enjoy competitive programming\n\nYou\'ve asked about many coding problems and usually want:'),
 Docume

### Simple use with LCEL

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

In [30]:
template="""Answer the question based only on the following context:
{context}
Question:{question}
"""
prompt=ChatPromptTemplate.from_template(template)
model= ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key
)

In [31]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

chain=(
    {"context":retriver|format_docs,"question":RunnablePassthrough()}
    |prompt
    |model
    |StrOutputParser()
)

In [32]:
chain.invoke("tell me about his projects")

'He prefers projects that impress recruiters and add resume value, rather than small demo projects.\n\nSome project ideas he has explored include:\n*   AI Interview Assistant\n*   Plant Disease Classifier\n*   Text/PDF to animated educational videos\n*   Deep learning projects for his resume'